# Prophet - Decomposable Time Series Forecasting
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/projects/forecasting/prophet_time_series.ipynb)

Prophet (Meta) fits trend + seasonality + holidays + regressors with a curve-fitting approach that is robust to missing data and outliers. Columns required: exactly `ds` (dates) and `y` (values).

Free on Colab; CPU is enough.

In [ ]:
!pip install -q prophet

## 1. Daily data with weekly + yearly patterns

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

rng = np.random.default_rng(7)
days = pd.date_range("2022-01-01", "2024-06-30", freq="D")
t = np.arange(len(days))
y = (100
     + 0.03 * t                                        # growth trend
     + 15 * np.sin(2 * np.pi * t / 365.25)             # yearly
     - 10 * (pd.Series(days.dayofweek >= 5))           # weekend dip
     + rng.normal(0, 4, len(days)))
df = pd.DataFrame({"ds": days, "y": y})
df.plot(x="ds", y="y", figsize=(11, 3.5), legend=False, title="Daily demand"); plt.show()

## 2. Fit + predict 180 days ahead

In [ ]:
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(df)

future = m.make_future_dataframe(periods=180)
fcst = m.predict(future)

m.plot(fcst, figsize=(11, 5)); plt.title("Forecast with uncertainty band"); plt.show()

## 3. Decompose: what drives the pattern?

In [ ]:
m.plot_components(fcst, figsize=(10, 7)); plt.show()

Reads directly: long-term trend, day-of-week effect, within-year cycle - each additive and interpretable.

## 4. Holidays & custom seasons

In [ ]:
playoffs = pd.DataFrame({"holiday": "promo", "ds": pd.to_datetime(
    ["2023-11-24", "2024-11-29"])})
m2 = Prophet(holidays=playoffs)
m2.fit(df)
print(m2.construct_holiday_dataframe("promo").head())

## When Prophet vs ARIMA/SARIMA
| Situation | Choose |
|---|---|
| strong multiple seasonalities, holidays, missing data | **Prophet** |
| pure autoregressive dynamics, need theory-grade baseline | **SARIMA** |
| many external features | gradient boosting on lag features |

Caveats: Prophet assumes smooth trends (add changepoints for shifts) and struggles with heavy regime changes - always benchmark against a naive seasonal average.